In [1]:
# Cell 1 — Imports
from dotenv import load_dotenv
import os
load_dotenv(r'C:\Users\Gurveer\ds-portfolio\.env')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

c:\Users\Gurveer\anaconda3\envs\ds-portfolio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful
PyTorch: 2.11.0+cpu
CUDA available: False


In [2]:
# Cell 2 — Build Mental Health Text Dataset
# Curated posts simulating Reddit mental health communities
# Labels: anxiety, isolation, depression, neutral

data = [
    # Anxiety
    {"text": "I can't stop worrying about everything. My heart races and I feel like something bad is always about to happen.", "label": "anxiety"},
    {"text": "I've been having panic attacks at work. I can't breathe and my hands shake uncontrollably.", "label": "anxiety"},
    {"text": "The constant fear of judgment from others is paralyzing me. I avoid social situations completely now.", "label": "anxiety"},
    {"text": "I keep replaying conversations in my head wondering if I said the wrong thing to everyone.", "label": "anxiety"},
    {"text": "My anxiety is so bad I can't sleep. I just lie awake catastrophizing about the future.", "label": "anxiety"},
    {"text": "I feel trapped in my own mind. The worry never stops, it just cycles through different fears.", "label": "anxiety"},
    {"text": "I canceled plans again because I was too anxious. I hate that this controls my life.", "label": "anxiety"},
    {"text": "Every morning I wake up with a pit in my stomach. Dread follows me everywhere I go.", "label": "anxiety"},
    {"text": "I'm terrified of making mistakes. The fear of failure is so overwhelming I can't start anything.", "label": "anxiety"},
    {"text": "Social anxiety has taken over. I rehearse conversations hours before having them.", "label": "anxiety"},
    {"text": "I feel on edge all the time. The slightest sound makes me jump and my body stays tense.", "label": "anxiety"},
    {"text": "Overthinking is destroying me. I can't make simple decisions without spiraling into what-ifs.", "label": "anxiety"},

    # Isolation
    {"text": "I haven't left my apartment in three days. I don't see the point in going outside anymore.", "label": "isolation"},
    {"text": "Nobody checks on me. I could disappear and no one would notice for weeks.", "label": "isolation"},
    {"text": "I sit alone every lunch break. I watch everyone else laugh and talk and I feel invisible.", "label": "isolation"},
    {"text": "My phone never rings. I've stopped expecting it to. The silence is deafening.", "label": "isolation"},
    {"text": "I moved to a new city for work and haven't made a single friend in six months.", "label": "isolation"},
    {"text": "I used to have friends. Now I eat dinner alone every night and wonder what happened.", "label": "isolation"},
    {"text": "Even in a crowded room I feel completely alone. Like there's glass between me and everyone else.", "label": "isolation"},
    {"text": "I turned down the invitation again. It's easier to stay home than to pretend I'm okay.", "label": "isolation"},
    {"text": "The loneliness is physical. It sits in my chest like a weight I can't lift.", "label": "isolation"},
    {"text": "I haven't had a real conversation in weeks. Just surface level exchanges with coworkers.", "label": "isolation"},
    {"text": "My family stopped calling. I think they gave up on me and honestly I understand why.", "label": "isolation"},
    {"text": "I spend weekends completely alone. Sunday evenings are the hardest. The dread of another week alone.", "label": "isolation"},

    # Depression
    {"text": "I can't get out of bed. Everything feels heavy and pointless. What's the use.", "label": "depression"},
    {"text": "I've lost interest in everything I used to love. Music, cooking, going outside. Nothing feels worth it.", "label": "depression"},
    {"text": "I feel empty inside. Not sad exactly, just completely hollow and numb all the time.", "label": "depression"},
    {"text": "I'm exhausted even after sleeping 12 hours. My body feels like it's made of concrete.", "label": "depression"},
    {"text": "I can't remember the last time I felt genuinely happy. It's been months of gray.", "label": "depression"},
    {"text": "I've been crying for no reason again. Or maybe for every reason. I can't tell anymore.", "label": "depression"},
    {"text": "Getting through a single day feels like climbing a mountain. I'm so tired of fighting myself.", "label": "depression"},
    {"text": "I stopped caring about my appearance, my work, my future. Nothing seems to matter anymore.", "label": "depression"},
    {"text": "The darkness comes in waves. Sometimes I float above it but it always pulls me back under.", "label": "depression"},
    {"text": "I feel like a burden to everyone around me. They'd be better off without having to worry.", "label": "depression"},
    {"text": "I can't concentrate on anything. Simple tasks take hours and I still can't finish them.", "label": "depression"},
    {"text": "I've been isolating because I don't want people to see how low I've gotten.", "label": "depression"},

    # Neutral / Positive
    {"text": "Had a great workout this morning. Feeling energized and ready to tackle the week.", "label": "neutral"},
    {"text": "Finally finished that project at work. My team did a fantastic job and I'm proud of us.", "label": "neutral"},
    {"text": "Went hiking with friends this weekend. The fresh air and good company did wonders.", "label": "neutral"},
    {"text": "Started meditating daily. It's been two weeks and I already notice a difference in my mood.", "label": "neutral"},
    {"text": "Cooked a new recipe tonight and it turned out amazing. Small wins matter.", "label": "neutral"},
    {"text": "Had a long talk with an old friend. It reminded me how important connection is.", "label": "neutral"},
    {"text": "Feeling grateful today. Good coffee, sunshine, and a manageable to-do list.", "label": "neutral"},
    {"text": "My therapy session was really productive today. Making real progress on my goals.", "label": "neutral"},
    {"text": "Got promoted at work. All the hard work finally paid off. I'm really happy.", "label": "neutral"},
    {"text": "Spent the evening reading a good book. Sometimes the quiet is exactly what I need.", "label": "neutral"},
    {"text": "Family dinner tonight. Everyone was laughing and catching up. Really needed that.", "label": "neutral"},
    {"text": "Feeling hopeful about the future. Taking things one day at a time and it's working.", "label": "neutral"},
]

df = pd.DataFrame(data)
print(f"Dataset size: {len(df)} posts")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nSample post:")
print(f"  [{df.iloc[0]['label']}] {df.iloc[0]['text'][:80]}...")

Dataset size: 48 posts

Label distribution:
label
anxiety       12
isolation     12
depression    12
neutral       12
Name: count, dtype: int64

Sample post:
  [anxiety] I can't stop worrying about everything. My heart races and I feel like something...


In [3]:
# Cell 3 — Zero-Shot Classification with HuggingFace
print("Loading zero-shot classifier (facebook/bart-large-mnli)...")
print("This may take 1-2 minutes on first run...")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1  # CPU
)

candidate_labels = ["anxiety", "isolation", "depression", "neutral"]

print("Classifying all posts...")
predictions = []
for i, row in df.iterrows():
    result = classifier(row['text'], candidate_labels)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    predictions.append({
        'text':           row['text'][:60] + '...',
        'actual':         row['label'],
        'predicted':      top_label,
        'confidence':     round(top_score, 4),
        'correct':        top_label == row['label']
    })
    if (i+1) % 10 == 0:
        print(f"  Processed {i+1}/{len(df)} posts...")

pred_df = pd.DataFrame(predictions)
accuracy = pred_df['correct'].mean()

print(f"\n=== Zero-Shot Classification Results ===")
print(f"Overall accuracy: {accuracy*100:.1f}%")
print(f"\nPer-label accuracy:")
for label in candidate_labels:
    subset = pred_df[pred_df['actual'] == label]
    acc    = subset['correct'].mean()
    print(f"  {label:<12} {acc*100:.1f}%")

Loading zero-shot classifier (facebook/bart-large-mnli)...
This may take 1-2 minutes on first run...


Device set to use cpu


Classifying all posts...
  Processed 10/48 posts...
  Processed 20/48 posts...
  Processed 30/48 posts...
  Processed 40/48 posts...

=== Zero-Shot Classification Results ===
Overall accuracy: 68.8%

Per-label accuracy:
  anxiety      83.3%
  isolation    100.0%
  depression   16.7%
  neutral      75.0%


In [4]:
# Cell 4 — Confidence Analysis & Misclassification Review
# Confidence distribution by label
fig1 = px.box(
    pred_df, x='actual', y='confidence',
    color='correct',
    title='Prediction Confidence by Label & Correctness',
    template='plotly_dark',
    color_discrete_map={True: 'seagreen', False: 'crimson'}
)
fig1.update_layout(width=800, height=450)
fig1.show()

# Confusion heatmap
from sklearn.metrics import confusion_matrix
import numpy as np

cm = confusion_matrix(
    pred_df['actual'], pred_df['predicted'],
    labels=candidate_labels
)
fig2 = go.Figure(data=go.Heatmap(
    z=cm,
    x=candidate_labels,
    y=candidate_labels,
    colorscale='Blues',
    text=cm, texttemplate="%{text}",
    showscale=True
))
fig2.update_layout(
    title='Confusion Matrix — Zero-Shot Mental Health Classifier',
    xaxis_title='Predicted',
    yaxis_title='Actual',
    template='plotly_dark',
    width=600, height=500
)
fig2.show()

# Show misclassifications
misses = pred_df[pred_df['correct'] == False]
print(f"\nMisclassifications: {len(misses)}/{len(pred_df)}")
print(misses[['actual','predicted','confidence','text']].to_string(index=False))


Misclassifications: 15/48
    actual predicted  confidence                                                            text
   anxiety isolation      0.7125 The constant fear of judgment from others is paralyzing me. ...
   anxiety isolation      0.9099 I feel trapped in my own mind. The worry never stops, it jus...
depression isolation      0.3442 I can't get out of bed. Everything feels heavy and pointless...
depression   neutral      0.3890 I've lost interest in everything I used to love. Music, cook...
depression   neutral      0.4241 I feel empty inside. Not sad exactly, just completely hollow...
depression isolation      0.3885 I'm exhausted even after sleeping 12 hours. My body feels li...
depression isolation      0.4335 I can't remember the last time I felt genuinely happy. It's ...
depression isolation      0.6449 I've been crying for no reason again. Or maybe for every rea...
depression isolation      0.8197 I stopped caring about my appearance, my work, my future. No...
dep

In [5]:
# Cell 5 — Longitudinal Trend Simulation
import datetime

# Simulate a user tracking their mental state over 30 days
np.random.seed(42)
n_days = 30

# Simulate realistic mental health trajectory
# Week 1: high anxiety, Week 2: depression onset, 
# Week 3: isolation, Week 4: gradual recovery
daily_posts = [
    {"day": 1,  "text": "Woke up with that familiar dread again. Heart racing before I even get out of bed.", "true_state": "anxiety"},
    {"day": 2,  "text": "Another panic attack at work. Had to hide in the bathroom for 20 minutes.", "true_state": "anxiety"},
    {"day": 3,  "text": "The worry is exhausting. I just want one day where my mind is quiet.", "true_state": "anxiety"},
    {"day": 4,  "text": "Called in sick. Couldn't face going in today. Just stayed in bed.", "true_state": "depression"},
    {"day": 5,  "text": "Haven't eaten much today. Everything feels pointless and heavy.", "true_state": "depression"},
    {"day": 6,  "text": "Turned down plans with friends again. I don't have the energy to pretend.", "true_state": "isolation"},
    {"day": 7,  "text": "One week of feeling awful. I don't know how much longer I can do this.", "true_state": "depression"},
    {"day": 8,  "text": "My phone hasn't rung in days. I think people have given up on me.", "true_state": "isolation"},
    {"day": 9,  "text": "Sat alone at lunch again. Watched everyone else and felt completely invisible.", "true_state": "isolation"},
    {"day": 10, "text": "I can't concentrate on anything. Simple tasks are taking me hours.", "true_state": "depression"},
    {"day": 11, "text": "Haven't left the apartment since Monday. The walls are closing in.", "true_state": "isolation"},
    {"day": 12, "text": "Cried for no reason again this morning. Or maybe for every reason.", "true_state": "depression"},
    {"day": 13, "text": "Feeling a tiny bit better today. Made myself shower and eat a real meal.", "true_state": "neutral"},
    {"day": 14, "text": "Two weeks of this. I made an appointment with a therapist finally.", "true_state": "neutral"},
    {"day": 15, "text": "First therapy session today. It was hard but I feel slightly less alone.", "true_state": "neutral"},
    {"day": 16, "text": "Still anxious but trying the breathing exercises my therapist suggested.", "true_state": "anxiety"},
    {"day": 17, "text": "Went for a short walk today. First time outside in days. Small win.", "true_state": "neutral"},
    {"day": 18, "text": "Texted an old friend. We're meeting for coffee next week.", "true_state": "neutral"},
    {"day": 19, "text": "Bad day. The dark thoughts came back hard. Struggling to stay present.", "true_state": "depression"},
    {"day": 20, "text": "Called my therapist between sessions. She helped me through the rough patch.", "true_state": "neutral"},
    {"day": 21, "text": "Three weeks in. Recovery isn't linear but I'm still here.", "true_state": "neutral"},
    {"day": 22, "text": "Had coffee with my friend. First real social interaction in weeks.", "true_state": "neutral"},
    {"day": 23, "text": "Anxiety still there but quieter today. Managed to finish a work project.", "true_state": "neutral"},
    {"day": 24, "text": "Cooked dinner for the first time in weeks. Such a small thing but it helped.", "true_state": "neutral"},
    {"day": 25, "text": "Had a setback today. Old isolation feelings crept back. Texted my therapist.", "true_state": "isolation"},
    {"day": 26, "text": "Better today. The setbacks are getting shorter. Progress is real.", "true_state": "neutral"},
    {"day": 27, "text": "Went to a group fitness class. Terrifying but I did it.", "true_state": "neutral"},
    {"day": 28, "text": "Starting to feel like myself again. Still work to do but hopeful.", "true_state": "neutral"},
    {"day": 29, "text": "Grateful for the small moments of peace. They're coming more often now.", "true_state": "neutral"},
    {"day": 30, "text": "One month tracked. Looking back I can see the progress. Therapy works.", "true_state": "neutral"},
]

trend_df = pd.DataFrame(daily_posts)

print("Running longitudinal classification...")
trend_results = []
for _, row in trend_df.iterrows():
    result = classifier(row['text'], candidate_labels)
    trend_results.append({
        'day':       row['day'],
        'text':      row['text'][:50] + '...',
        'predicted': result['labels'][0],
        'confidence':round(result['scores'][0], 3),
        'true_state':row['true_state']
    })
    if row['day'] % 5 == 0:
        print(f"  Day {row['day']} processed...")

trend_results_df = pd.DataFrame(trend_results)

# Encode for plotting
state_colors = {
    'anxiety':    'orange',
    'isolation':  'royalblue',
    'depression': 'crimson',
    'neutral':    'seagreen'
}

fig = go.Figure()
for state, color in state_colors.items():
    mask = trend_results_df['predicted'] == state
    fig.add_trace(go.Scatter(
        x=trend_results_df[mask]['day'],
        y=trend_results_df[mask]['confidence'],
        mode='markers',
        name=state,
        marker=dict(color=color, size=12),
        text=trend_results_df[mask]['text'],
        hovertemplate='Day %{x}<br>Confidence: %{y}<br>%{text}'
    ))

fig.update_layout(
    title='30-Day Mental Health Trend — Longitudinal Tracking',
    xaxis_title='Day',
    yaxis_title='Classification Confidence',
    template='plotly_dark',
    width=900, height=500
)
fig.show()
print("\nLongitudinal tracking complete")

Running longitudinal classification...
  Day 5 processed...
  Day 10 processed...
  Day 15 processed...
  Day 20 processed...
  Day 25 processed...
  Day 30 processed...



Longitudinal tracking complete


In [6]:
# Cell 6 — Weekly Summary & Export
# Weekly rollup
trend_results_df['week'] = ((trend_results_df['day'] - 1) // 7) + 1
weekly = trend_results_df.groupby(['week','predicted']).size().reset_index(name='count')

fig2 = px.bar(
    weekly, x='week', y='count', color='predicted',
    title='Weekly Mental State Distribution — 30-Day Tracking',
    template='plotly_dark',
    color_discrete_map=state_colors,
    barmode='stack',
    labels={'week': 'Week', 'count': 'Posts', 'predicted': 'State'}
)
fig2.update_layout(width=800, height=450)
fig2.show()

print("=== 30-Day Summary ===")
print(f"\nMost common state:  {trend_results_df['predicted'].mode()[0]}")
print(f"\nState distribution:")
print(trend_results_df['predicted'].value_counts().to_string())
print(f"\nWeek-by-week dominant state:")
for week in [1,2,3,4]:
    week_data = trend_results_df[trend_results_df['week']==week]
    dominant  = week_data['predicted'].mode()[0]
    print(f"  Week {week}: {dominant}")

# Export
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-25-anxiety-sentiment-tracker\outputs'
os.makedirs(output_dir, exist_ok=True)
pred_df.to_csv(f'{output_dir}\\classification_results.csv', index=False)
trend_results_df.to_csv(f'{output_dir}\\longitudinal_tracking.csv', index=False)
print(f"\nExports saved to outputs/")

=== 30-Day Summary ===

Most common state:  isolation

State distribution:
predicted
isolation     16
neutral        7
anxiety        6
depression     1

Week-by-week dominant state:
  Week 1: isolation
  Week 2: isolation
  Week 3: isolation
  Week 4: isolation

Exports saved to outputs/
